# S6E4 Predicting Irrigation Need
## Score: .96063

In [1]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", 200)


In [2]:
PROJECT_ROOT = Path(".")
DATA_DIR = PROJECT_ROOT / "playground-series-s6e4"
ORIGINAL_PATH = PROJECT_ROOT / "original-data" / "irrigation_prediction.csv"
ANCHOR_SUB_PATH = PROJECT_ROOT / "score97131.csv"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SUB_PATH = PROJECT_ROOT / "submission.csv"

TARGET = "Irrigation_Need"
ID_COL = "id"
SOURCE_COL = "_source"
BASE_SEED = 42
OVERNIGHT_MODE = False
SEED_LIST = [42, 1337, 2026]
N_SPLITS = 5
STACKER_SPLITS = 5
USE_ORIGINAL_DATA = True
ORIGINAL_ROW_WEIGHT = 0.38
ANCHOR_BLEND_WEIGHT = 0.05
USE_ENGINEERED_FEATURES = True
USE_INTERACTION_FEATURES = True
USE_CAT2 = True
USE_TARGET_ENCODING = True
TE_N_SPLITS = 5
TE_SMOOTHING = 25.0
USE_FREQ_ENCODING = True
USE_HGB = True
USE_EXTRA_TREES = False
USE_RANDOM_FOREST = True
USE_MLP = True
USE_LGBM_META_BLEND = True
USE_PSEUDO_LABEL = True
PSEUDO_CONF_MIN = 0.989
PSEUDO_MAX_PER_CLASS = 6500
PSEUDO_ROW_WEIGHT = 0.25
USE_ANCHOR_FALLBACK = False
FALLBACK_CONF_THRESHOLD = 0.60
SUBMISSION_EXTRA_BLENDS = []



if not TRAIN_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        f"Expected data files under {DATA_DIR.resolve()}"
    )

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

train_df[SOURCE_COL] = "kaggle"

if USE_ORIGINAL_DATA and ORIGINAL_PATH.exists():
    original_df = pd.read_csv(ORIGINAL_PATH)
    original_df = original_df[[c for c in train_df.columns if c not in [ID_COL, SOURCE_COL]]]
    original_df[SOURCE_COL] = "original"
    train_df = pd.concat([train_df, original_df], axis=0, ignore_index=True)
    print("original shape:", original_df.shape)
else:
    print("original data disabled or missing")

print("train shape:", train_df.shape)
print("test shape :", test_df.shape)
train_df.head()


original shape: (10000, 21)
train shape: (640000, 22)
test shape : (270000, 20)


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need,_source
0,0.0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low,kaggle
1,1.0,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low,kaggle
2,2.0,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low,kaggle
3,3.0,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium,kaggle
4,4.0,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low,kaggle


In [3]:
display(train_df[TARGET].value_counts(dropna=False))
print("\nTarget distribution (%):")
display((train_df[TARGET].value_counts(normalize=True) * 100).round(2))

missing_train = train_df.isna().mean().sort_values(ascending=False)
missing_test = test_df.isna().mean().sort_values(ascending=False)

print("\nTop missing-rate features (train):")
display(missing_train.head(10))
print("\nTop missing-rate features (test):")
display(missing_test.head(10))

Irrigation_Need
Low       375781
Medium    242874
High       21345
Name: count, dtype: int64


Target distribution (%):


Irrigation_Need
Low       58.72
Medium    37.95
High       3.34
Name: proportion, dtype: float64


Top missing-rate features (train):


id                         0.015625
Soil_Type                  0.000000
Soil_pH                    0.000000
Soil_Moisture              0.000000
Organic_Carbon             0.000000
Electrical_Conductivity    0.000000
Temperature_C              0.000000
Humidity                   0.000000
Rainfall_mm                0.000000
Sunlight_Hours             0.000000
dtype: float64


Top missing-rate features (test):


id                         0.0
Soil_Type                  0.0
Soil_pH                    0.0
Soil_Moisture              0.0
Organic_Carbon             0.0
Electrical_Conductivity    0.0
Temperature_C              0.0
Humidity                   0.0
Rainfall_mm                0.0
Sunlight_Hours             0.0
dtype: float64

In [4]:



def add_engineered_features(df):
    out = df.copy()
    eps = 1e-6
    out["feat_rain_per_moisture"] = out["Rainfall_mm"] / (out["Soil_Moisture"] + eps)
    out["feat_humid_temp"] = out["Humidity"] * out["Temperature_C"] / 100.0
    out["feat_prev_per_area"] = out["Previous_Irrigation_mm"] / (out["Field_Area_hectare"] + eps)
    out["feat_sun_wind"] = out["Sunlight_Hours"] * out["Wind_Speed_kmh"] / 100.0
    out["feat_oc_ph"] = out["Organic_Carbon"] * out["Soil_pH"]
    if USE_INTERACTION_FEATURES:
        out["cross_crop_season"] = (
            out["Crop_Type"].astype(str) + "_" + out["Season"].astype(str)
        )
        out["cross_irrig_water"] = (
            out["Irrigation_Type"].astype(str) + "_" + out["Water_Source"].astype(str)
        )
        out["cross_region_crop"] = (
            out["Region"].astype(str) + "_" + out["Crop_Type"].astype(str)
        )
    return out


def _fit_te_map(cat_series, y_series, classes, priors, m):
    df = pd.DataFrame({"c": cat_series.values, "y": y_series.values})
    te_map = {}
    for cval, grp in df.groupby("c", sort=False):
        n = len(grp)
        counts = grp["y"].value_counts()
        arr = np.array([counts.get(cl, 0) for cl in classes], dtype=np.float64)
        te_map[cval] = (arr + m * priors) / (n + m)
    return te_map, priors.copy()


def _apply_te_rows(series, te_map, default, n_classes):
    out = np.zeros((len(series), n_classes), dtype=np.float64)
    vals = series.values
    for i in range(len(series)):
        out[i] = te_map.get(vals[i], default)
    return out


def add_multiclass_target_encoding_oof(
    train_df, test_df, target_col, cat_cols, seed, n_splits, m
):
    y = train_df[target_col]
    classes = sorted(y.unique().tolist())
    n_classes = len(classes)
    priors = y.value_counts(normalize=True).reindex(classes).fillna(0).values.astype(np.float64)
    oof_mat = np.zeros((len(train_df), len(cat_cols) * n_classes), dtype=np.float64)
    col_off = {c: i * n_classes for i, c in enumerate(cat_cols)}
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for tr_idx, va_idx in skf.split(train_df, y):
        y_tr = y.iloc[tr_idx]
        for col in cat_cols:
            te_map, default = _fit_te_map(
                train_df[col].iloc[tr_idx], y_tr, classes, priors, m
            )
            block = _apply_te_rows(
                train_df[col].iloc[va_idx], te_map, default, n_classes
            )
            sl = slice(col_off[col], col_off[col] + n_classes)
            oof_mat[va_idx, sl] = block
    test_mat = np.zeros((len(test_df), len(cat_cols) * n_classes), dtype=np.float64)
    for col in cat_cols:
        te_map, default = _fit_te_map(train_df[col], y, classes, priors, m)
        test_mat[:, col_off[col] : col_off[col] + n_classes] = _apply_te_rows(
            test_df[col], te_map, default, n_classes
        )
    for i, col in enumerate(cat_cols):
        for k, cls in enumerate(classes):
            name = "te_" + str(col).replace(" ", "_") + "_" + str(cls).replace(" ", "_")
            train_df[name] = oof_mat[:, i * n_classes + k]
            test_df[name] = test_mat[:, i * n_classes + k]
    print(
        f"Target encoding: added {len(cat_cols) * n_classes} columns "
        f"(splits={n_splits}, m={m})"
    )
    return train_df, test_df


def add_frequency_encoding_oof(
    train_df, test_df, target_col, cat_cols, seed, n_splits, smooth=10.0
):
    y = train_df[target_col]
    n = len(train_df)
    n_cat = len(cat_cols)
    oof = np.zeros((n, n_cat), dtype=np.float64)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed + 11)
    for tr_idx, va_idx in skf.split(train_df, y):
        tr_df = train_df.iloc[tr_idx]
        for j, col in enumerate(cat_cols):
            vc = tr_df[col].value_counts()
            va_vals = train_df[col].iloc[va_idx]
            oof[va_idx, j] = np.log1p(
                va_vals.map(lambda x: float(vc.get(x, 0)) + smooth).values
            )
    for j, col in enumerate(cat_cols):
        vc = train_df[col].value_counts()
        cname = "freq_log_" + str(col).replace(" ", "_")
        train_df[cname] = oof[:, j]
        test_df[cname] = np.log1p(
            test_df[col].map(lambda x: float(vc.get(x, 0)) + smooth).values
        )
    print(f"Frequency encoding: added {n_cat} columns (splits={n_splits})")
    return train_df, test_df


if USE_ENGINEERED_FEATURES:
    train_df = add_engineered_features(train_df)
    test_df = add_engineered_features(test_df)

pre_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL, SOURCE_COL]]
_cat_for_te = [c for c in pre_cols if train_df[c].dtype == "object"]
if USE_TARGET_ENCODING and len(_cat_for_te) > 0:
    train_df, test_df = add_multiclass_target_encoding_oof(
        train_df,
        test_df,
        TARGET,
        _cat_for_te,
        BASE_SEED,
        TE_N_SPLITS,
        TE_SMOOTHING,
    )

_obj_cols = [
    c
    for c in train_df.columns
    if c not in [TARGET, ID_COL, SOURCE_COL] and train_df[c].dtype == "object"
]
if USE_FREQ_ENCODING and len(_obj_cols) > 0:
    train_df, test_df = add_frequency_encoding_oof(
        train_df, test_df, TARGET, _obj_cols, BASE_SEED, TE_N_SPLITS, 10.0
    )

TRAIN_DF_BASE = train_df.copy()

y = train_df[TARGET].copy()
source_flag = train_df[SOURCE_COL].copy()
X = train_df.drop(columns=[TARGET]).copy()
X_test = test_df.copy()

feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
X = X[feature_cols]
X_test = X_test[feature_cols]

cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Total features: {len(feature_cols)}")
print(f"Categorical   : {len(cat_cols)} -> {cat_cols}")
print(f"Numerical     : {len(num_cols)}")
print("Source distribution:")
display(source_flag.value_counts())


Target encoding: added 33 columns (splits=5, m=25.0)
Frequency encoding: added 11 columns (splits=5)
Total features: 71
Categorical   : 11 -> ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region', 'cross_crop_season', 'cross_irrig_water', 'cross_region_crop']
Numerical     : 60
Source distribution:


_source
kaggle      630000
original     10000
Name: count, dtype: int64

In [5]:
classes = sorted(TRAIN_DF_BASE[TARGET].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}
idx_to_class = {v: k for k, v in class_to_idx.items()}

if OVERNIGHT_MODE:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 1200, 0.04, 8, 120, 200
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 1200, 0.035, 96, 120
    xgb_est, xgb_lr, xgb_depth, xgb_es = 1200, 0.035, 8, 120
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 900, 0.05, 6, 100, 150
    hgb_max_iter, et_n_est, mlp_hidden, rf_n_est = 700, 700, (192, 96), 500
else:
    cat_iters, cat_lr, cat_depth, cat_es, cat_verbose = 700, 0.06, 7, 80, 100
    lgb_est, lgb_lr, lgb_leaves, lgb_es = 800, 0.04, 63, 80
    xgb_est, xgb_lr, xgb_depth, xgb_es = 800, 0.04, 7, 80
    cat2_iters, cat2_lr, cat2_depth, cat2_es, cat2_verbose = 500, 0.08, 6, 80, 100
    hgb_max_iter, et_n_est, mlp_hidden, rf_n_est = 380, 400, (128, 64), 400


def build_meta_features(*parts):
    eps = 1e-15
    parts = tuple(parts)
    blocks = [np.hstack(parts)]
    for p in parts:
        blocks.append(np.log(np.clip(p, eps, 1.0)))
    for p in parts:
        ent = -(p * np.log(np.clip(p, eps, 1.0))).sum(axis=1, keepdims=True)
        blocks.append(ent)
    for p in parts:
        blocks.append(np.max(p, axis=1, keepdims=True))
    for p in parts:
        ps = np.sort(p, axis=1)
        margin = (ps[:, -1] - ps[:, -2]).reshape(-1, 1)
        blocks.append(margin)
    return np.hstack(blocks)


def build_pseudo_augmented_train(train_df_base, test_df, test_probs):
    rng = np.random.RandomState(BASE_SEED + 99)
    conf = test_probs.max(axis=1)
    pred_i = np.argmax(test_probs, axis=1)
    keep_idx = []
    for cls_idx in range(len(classes)):
        mask = (conf >= PSEUDO_CONF_MIN) & (pred_i == cls_idx)
        ix = np.where(mask)[0]
        if len(ix) > PSEUDO_MAX_PER_CLASS:
            ix = rng.choice(ix, PSEUDO_MAX_PER_CLASS, replace=False)
        keep_idx.extend(ix.tolist())
    keep_idx = np.unique(np.array(keep_idx, dtype=int))
    pseudo = test_df.iloc[keep_idx].copy()
    pseudo[TARGET] = [idx_to_class[pred_i[i]] for i in keep_idx]
    pseudo[SOURCE_COL] = "pseudo"
    out = pd.concat([train_df_base, pseudo], axis=0, ignore_index=True)
    print(
        f"Pseudo-label: added {len(pseudo)} rows "
        f"(conf>={PSEUDO_CONF_MIN}, cap={PSEUDO_MAX_PER_CLASS}/class)"
    )
    return out


def temper_probs(p, T):
    z = np.log(np.clip(p, 1e-15, 1.0))
    zs = z / T
    ex = np.exp(zs - zs.max(axis=1, keepdims=True))
    return ex / ex.sum(axis=1, keepdims=True)


def build_sklearn_matrix(X_df, cat_cols, num_cols, medians, ord_enc, fit=False):
    Xn = X_df[num_cols].copy()
    for c in num_cols:
        Xn[c] = Xn[c].fillna(medians[c])
    Xcat = X_df[cat_cols].astype(str).fillna("__nan__")
    if fit:
        oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
        oc = oe.fit_transform(Xcat)
    else:
        oe = ord_enc
        oc = oe.transform(Xcat)
    xm = np.ascontiguousarray(
        np.hstack([Xn.values.astype(np.float64), oc.astype(np.float64)])
    )
    return xm, oe


def run_training_phase(train_df_run, phase_name):
    run_start = time.perf_counter()
    print(f"\n{'='*20} {phase_name} {'='*20}")
    print(
        f"Config: OVERNIGHT_MODE={OVERNIGHT_MODE} | seeds={len(SEED_LIST)} | "
        f"n_splits={N_SPLITS} | stacker_splits={STACKER_SPLITS} | "
        f"HGB={USE_HGB} ET={USE_EXTRA_TREES} RF={USE_RANDOM_FOREST} "
        f"CAT2={USE_CAT2} MLP={USE_MLP} LGBM_META={USE_LGBM_META_BLEND}"
    )

    y = train_df_run[TARGET].copy()
    source_flag = train_df_run[SOURCE_COL].copy()
    X = train_df_run.drop(columns=[TARGET]).copy()
    X_test = test_df.copy()
    feature_cols = [c for c in X.columns if c not in [ID_COL, SOURCE_COL]]
    X = X[feature_cols]
    X_test = X_test[feature_cols]
    cat_cols = [c for c in feature_cols if X[c].dtype == "object"]
    num_cols = [c for c in feature_cols if c not in cat_cols]

    y_idx = y.map(class_to_idx).astype(int)
    class_counts = y.value_counts()
    class_weight_label = {
        cls: len(y) / (len(classes) * class_counts[cls])
        for cls in classes
    }
    class_weight_idx = {
        class_to_idx[cls]: weight
        for cls, weight in class_weight_label.items()
    }
    source_weight_map = {
        "kaggle": 1.0,
        "original": ORIGINAL_ROW_WEIGHT,
        "pseudo": PSEUDO_ROW_WEIGHT,
    }

    oof_cat = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_cat2 = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_lgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_xgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_hgb = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_et = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_mlp = np.zeros((len(X), len(classes)), dtype=np.float64)
    oof_rf = np.zeros((len(X), len(classes)), dtype=np.float64)

    test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_hgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_et = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_mlp = np.zeros((len(X_test), len(classes)), dtype=np.float64)
    test_rf = np.zeros((len(X_test), len(classes)), dtype=np.float64)

    fold_scores = []

    for seed_idx, seed in enumerate(SEED_LIST, start=1):
        seed_start = time.perf_counter()
        print(f"\n######## Seed {seed_idx}/{len(SEED_LIST)} (seed={seed}) ########")

        cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)

        seed_test_cat = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_cat2 = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_lgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_xgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_hgb = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_et = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_mlp = np.zeros((len(X_test), len(classes)), dtype=np.float64)
        seed_test_rf = np.zeros((len(X_test), len(classes)), dtype=np.float64)

        for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y), start=1):
            fold_start = time.perf_counter()
            print(f"\n=== Seed {seed} | Fold {fold}/{N_SPLITS} started ===")

            X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            y_train_idx, y_valid_idx = y_idx.iloc[train_idx], y_idx.iloc[valid_idx]
            source_train = source_flag.iloc[train_idx]

            for col in cat_cols:
                X_train[col] = X_train[col].astype("category")
                X_valid[col] = X_valid[col].astype("category")

            X_test_fold = X_test.copy()
            for col in cat_cols:
                X_test_fold[col] = X_test_fold[col].astype("category")

            sample_weight = (
                y_train.map(class_weight_label).values
                * source_train.map(source_weight_map).values
            )

            medians = X_train[num_cols].median()

            cat_model = CatBoostClassifier(
                iterations=cat_iters,
                learning_rate=cat_lr,
                depth=cat_depth,
                loss_function="MultiClass",
                eval_metric="TotalF1",
                random_seed=seed + fold,
                verbose=cat_verbose,
                thread_count=-1,
                class_weights=class_weight_label,
            )

            lgb_model = LGBMClassifier(
                n_estimators=lgb_est,
                learning_rate=lgb_lr,
                num_leaves=lgb_leaves,
                min_child_samples=40,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="multiclass",
                random_state=seed + fold,
                class_weight=class_weight_idx,
                verbosity=-1,
            )

            xgb_model = XGBClassifier(
                n_estimators=xgb_est,
                learning_rate=xgb_lr,
                max_depth=xgb_depth,
                min_child_weight=3,
                subsample=0.9,
                colsample_bytree=0.9,
                reg_lambda=1.0,
                objective="multi:softprob",
                num_class=len(classes),
                eval_metric="mlogloss",
                tree_method="hist",
                enable_categorical=True,
                random_state=seed + fold,
                n_jobs=-1,
            )

            print(f"Seed {seed} Fold {fold}: training CatBoost...")
            cat_model.fit(
                X_train,
                y_train,
                sample_weight=sample_weight,
                cat_features=cat_cols,
                eval_set=(X_valid, y_valid),
                use_best_model=True,
                early_stopping_rounds=cat_es,
            )

            if USE_CAT2:
                cat2_model = CatBoostClassifier(
                    iterations=cat2_iters,
                    learning_rate=cat2_lr,
                    depth=cat2_depth,
                    loss_function="MultiClass",
                    eval_metric="TotalF1",
                    random_seed=seed + fold + 777,
                    verbose=cat2_verbose,
                    thread_count=-1,
                    class_weights=class_weight_label,
                )
                print(f"Seed {seed} Fold {fold}: training CatBoost-2...")
                cat2_model.fit(
                    X_train,
                    y_train,
                    sample_weight=sample_weight,
                    cat_features=cat_cols,
                    eval_set=(X_valid, y_valid),
                    use_best_model=True,
                    early_stopping_rounds=cat2_es,
                )

            print(f"Seed {seed} Fold {fold}: training LightGBM...")
            lgb_model.fit(
                X_train,
                y_train_idx,
                sample_weight=sample_weight,
                eval_set=[(X_valid, y_valid_idx)],
                eval_metric="multi_logloss",
                callbacks=[early_stopping(stopping_rounds=lgb_es), log_evaluation(period=0)],
            )

            print(f"Seed {seed} Fold {fold}: training XGBoost...")
            xgb_model.fit(
                X_train,
                y_train_idx,
                eval_set=[(X_valid, y_valid_idx)],
                sample_weight=sample_weight,
                verbose=False,
            )

            X_tr_m, oe = build_sklearn_matrix(
                X_train, cat_cols, num_cols, medians, None, fit=True
            )
            X_va_m = build_sklearn_matrix(
                X_valid, cat_cols, num_cols, medians, oe, fit=False
            )[0]
            X_te_m = build_sklearn_matrix(
                X_test_fold, cat_cols, num_cols, medians, oe, fit=False
            )[0]

            hgb_model = None
            et_model = None
            mlp_model = None
            mlp_scaler = None

            if USE_HGB:
                print(f"Seed {seed} Fold {fold}: training HistGradientBoosting...")
                hgb_model = HistGradientBoostingClassifier(
                    max_iter=hgb_max_iter,
                    learning_rate=0.055,
                    max_depth=9,
                    l2_regularization=0.35,
                    random_state=seed + fold,
                    class_weight="balanced",
                )
                hgb_model.fit(X_tr_m, y_train_idx, sample_weight=sample_weight)

            if USE_EXTRA_TREES:
                print(f"Seed {seed} Fold {fold}: training ExtraTrees...")
                et_model = ExtraTreesClassifier(
                    n_estimators=et_n_est,
                    max_depth=18,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    random_state=seed + fold,
                    class_weight="balanced",
                    n_jobs=-1,
                )
                et_model.fit(X_tr_m, y_train_idx, sample_weight=sample_weight)

            if USE_MLP:
                print(f"Seed {seed} Fold {fold}: training MLP...")
                mlp_scaler = StandardScaler()
                X_tr_s = mlp_scaler.fit_transform(X_tr_m)
                X_va_s = mlp_scaler.transform(X_va_m)
                mlp_model = MLPClassifier(
                    hidden_layer_sizes=mlp_hidden,
                    max_iter=500,
                    early_stopping=True,
                    validation_fraction=0.12,
                    n_iter_no_change=25,
                    random_state=seed + fold,
                    alpha=1e-4,
                )
                mlp_model.fit(X_tr_s, y_train_idx, sample_weight=sample_weight)

            rf_model = None
            if USE_RANDOM_FOREST:
                print(f"Seed {seed} Fold {fold}: training RandomForest...")
                rf_model = RandomForestClassifier(
                    n_estimators=rf_n_est,
                    max_depth=20,
                    min_samples_leaf=2,
                    max_features="sqrt",
                    random_state=seed + fold + 333,
                    class_weight="balanced",
                    n_jobs=-1,
                )
                rf_model.fit(X_tr_m, y_train_idx, sample_weight=sample_weight)

            print(f"Seed {seed} Fold {fold}: model training complete")

            cat_valid = cat_model.predict_proba(X_valid)
            if USE_CAT2:
                cat2_valid = cat2_model.predict_proba(X_valid)
            else:
                cat2_valid = None
            lgb_valid = lgb_model.predict_proba(X_valid)
            xgb_valid = xgb_model.predict_proba(X_valid)

            hgb_valid = hgb_model.predict_proba(X_va_m) if USE_HGB else None
            et_valid = et_model.predict_proba(X_va_m) if USE_EXTRA_TREES else None
            if USE_MLP:
                mlp_valid = mlp_model.predict_proba(mlp_scaler.transform(X_va_m))
            else:
                mlp_valid = None

            rf_valid = rf_model.predict_proba(X_va_m) if USE_RANDOM_FOREST else None

            oof_cat[valid_idx] += cat_valid / len(SEED_LIST)
            if USE_CAT2:
                oof_cat2[valid_idx] += cat2_valid / len(SEED_LIST)
            oof_lgb[valid_idx] += lgb_valid / len(SEED_LIST)
            oof_xgb[valid_idx] += xgb_valid / len(SEED_LIST)
            if USE_HGB:
                oof_hgb[valid_idx] += hgb_valid / len(SEED_LIST)
            if USE_EXTRA_TREES:
                oof_et[valid_idx] += et_valid / len(SEED_LIST)
            if USE_MLP:
                oof_mlp[valid_idx] += mlp_valid / len(SEED_LIST)
            if USE_RANDOM_FOREST:
                oof_rf[valid_idx] += rf_valid / len(SEED_LIST)

            cat_pred = np.argmax(cat_valid, axis=1)
            lgb_pred = np.argmax(lgb_valid, axis=1)
            xgb_pred = np.argmax(xgb_valid, axis=1)

            cat_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in cat_pred])
            lgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in lgb_pred])
            xgb_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in xgb_pred])
            extra_scores = []
            if USE_CAT2:
                cat2_pred = np.argmax(cat2_valid, axis=1)
                cat2_score = balanced_accuracy_score(
                    y_valid, [idx_to_class[i] for i in cat2_pred]
                )
                extra_scores.append(f"cat2={cat2_score:.6f}")
            if USE_HGB:
                hp = np.argmax(hgb_valid, axis=1)
                extra_scores.append(f"hgb={balanced_accuracy_score(y_valid, [idx_to_class[i] for i in hp]):.6f}")
            if USE_EXTRA_TREES:
                ep = np.argmax(et_valid, axis=1)
                extra_scores.append(f"et={balanced_accuracy_score(y_valid, [idx_to_class[i] for i in ep]):.6f}")
            if USE_MLP:
                mp = np.argmax(mlp_valid, axis=1)
                extra_scores.append(f"mlp={balanced_accuracy_score(y_valid, [idx_to_class[i] for i in mp]):.6f}")
            if USE_RANDOM_FOREST:
                rp = np.argmax(rf_valid, axis=1)
                extra_scores.append(
                    f"rf={balanced_accuracy_score(y_valid, [idx_to_class[i] for i in rp]):.6f}"
                )

            msg = (
                f"Seed {seed} Fold {fold}: cat={cat_score:.6f} "
                f"lgb={lgb_score:.6f} xgb={xgb_score:.6f}"
            )
            if extra_scores:
                msg += " " + " ".join(extra_scores)
            print(msg)

            blend_parts = [cat_valid, lgb_valid, xgb_valid]
            if USE_CAT2:
                blend_parts.append(cat2_valid)
            if USE_HGB:
                blend_parts.append(hgb_valid)
            if USE_EXTRA_TREES:
                blend_parts.append(et_valid)
            if USE_MLP:
                blend_parts.append(mlp_valid)
            if USE_RANDOM_FOREST:
                blend_parts.append(rf_valid)
            blend_equal = np.mean(np.stack(blend_parts, axis=0), axis=0)
            blend_pred = np.argmax(blend_equal, axis=1)
            fold_score = balanced_accuracy_score(y_valid, [idx_to_class[i] for i in blend_pred])
            fold_scores.append(fold_score)
            fold_minutes = (time.perf_counter() - fold_start) / 60.0
            print(f"Seed {seed} Fold {fold}: equal-blend balanced_accuracy = {fold_score:.6f}")
            print(f"Seed {seed} Fold {fold}: completed in {fold_minutes:.2f} minutes")

            seed_test_cat += cat_model.predict_proba(X_test_fold)
            if USE_CAT2:
                seed_test_cat2 += cat2_model.predict_proba(X_test_fold)
            seed_test_lgb += lgb_model.predict_proba(X_test_fold)
            seed_test_xgb += xgb_model.predict_proba(X_test_fold)
            if USE_HGB:
                seed_test_hgb += hgb_model.predict_proba(X_te_m)
            if USE_EXTRA_TREES:
                seed_test_et += et_model.predict_proba(X_te_m)
            if USE_MLP:
                seed_test_mlp += mlp_model.predict_proba(mlp_scaler.transform(X_te_m))
            if USE_RANDOM_FOREST:
                seed_test_rf += rf_model.predict_proba(X_te_m)

        test_cat += seed_test_cat / N_SPLITS
        test_cat2 += seed_test_cat2 / N_SPLITS
        test_lgb += seed_test_lgb / N_SPLITS
        test_xgb += seed_test_xgb / N_SPLITS
        if USE_HGB:
            test_hgb += seed_test_hgb / N_SPLITS
        if USE_EXTRA_TREES:
            test_et += seed_test_et / N_SPLITS
        if USE_MLP:
            test_mlp += seed_test_mlp / N_SPLITS
        if USE_RANDOM_FOREST:
            test_rf += seed_test_rf / N_SPLITS

        print(
            f"Seed {seed} finished in {(time.perf_counter() - seed_start) / 60.0:.2f} minutes"
        )

    test_cat /= len(SEED_LIST)
    test_cat2 /= len(SEED_LIST)
    test_lgb /= len(SEED_LIST)
    test_xgb /= len(SEED_LIST)
    if USE_HGB:
        test_hgb /= len(SEED_LIST)
    if USE_EXTRA_TREES:
        test_et /= len(SEED_LIST)
    if USE_MLP:
        test_mlp /= len(SEED_LIST)
    if USE_RANDOM_FOREST:
        test_rf /= len(SEED_LIST)

    print("\nAll folds done. Building meta-features + cross-fitted OOF stacking...")

    parts_oof = [oof_cat, oof_lgb, oof_xgb]
    parts_test = [test_cat, test_lgb, test_xgb]
    if USE_CAT2:
        parts_oof.append(oof_cat2)
        parts_test.append(test_cat2)
    if USE_HGB:
        parts_oof.append(oof_hgb)
        parts_test.append(test_hgb)
    if USE_EXTRA_TREES:
        parts_oof.append(oof_et)
        parts_test.append(test_et)
    if USE_MLP:
        parts_oof.append(oof_mlp)
        parts_test.append(test_mlp)
    if USE_RANDOM_FOREST:
        parts_oof.append(oof_rf)
        parts_test.append(test_rf)

    meta_oof = build_meta_features(*parts_oof)
    meta_test = build_meta_features(*parts_test)
    print(f"Meta feature columns: {meta_oof.shape[1]}")

    best_stack_score = -1.0
    best_stack_c = None
    best_stack_proba_oof = None
    best_stack_proba_test = None

    stack_cv = StratifiedKFold(n_splits=STACKER_SPLITS, shuffle=True, random_state=BASE_SEED)

    for c in [0.25, 0.35, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0]:
        print(f"Stacking search: cross-fitting LogisticRegression C={c}")

        cv_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
        cv_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)

        for stack_fold, (st_tr_idx, st_va_idx) in enumerate(stack_cv.split(meta_oof, y_idx), start=1):
            X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
            y_st_tr = y_idx.iloc[st_tr_idx]

            stacker = Pipeline(
                [
                    ("scaler", StandardScaler()),
                    (
                        "clf",
                        LogisticRegression(
                            C=c,
                            max_iter=800,
                            solver="lbfgs",
                            class_weight="balanced",
                        ),
                    ),
                ]
            )
            stacker.fit(X_st_tr, y_st_tr)

            cv_meta_oof[st_va_idx] = stacker.predict_proba(X_st_va)
            cv_meta_test += stacker.predict_proba(meta_test)

            if stack_fold % 2 == 0:
                print(f"Stacking C={c}: finished meta-fold {stack_fold}/{STACKER_SPLITS}")

        cv_meta_test /= STACKER_SPLITS

        pred_oof = np.argmax(cv_meta_oof, axis=1)
        pred_oof_labels = [idx_to_class[i] for i in pred_oof]
        score = balanced_accuracy_score(y, pred_oof_labels)
        print(f"Stacking C={c}: cross-fitted OOF balanced_accuracy = {score:.6f}")

        if score > best_stack_score:
            best_stack_score = score
            best_stack_c = c
            best_stack_proba_oof = cv_meta_oof
            best_stack_proba_test = cv_meta_test

    print(f"Best stacker C: {best_stack_c} with OOF score {best_stack_score:.6f}")
    if USE_LGBM_META_BLEND:
        print("Cross-fitting LightGBM meta-stacker on meta-features...")
        lgb_meta_oof = np.zeros((len(meta_oof), len(classes)), dtype=np.float64)
        lgb_meta_test = np.zeros((len(meta_test), len(classes)), dtype=np.float64)
        for stack_fold, (st_tr_idx, st_va_idx) in enumerate(
            stack_cv.split(meta_oof, y_idx), start=1
        ):
            X_st_tr, X_st_va = meta_oof[st_tr_idx], meta_oof[st_va_idx]
            y_st_tr = y_idx.iloc[st_tr_idx]
            lgb_m = LGBMClassifier(
                n_estimators=420,
                learning_rate=0.045,
                num_leaves=48,
                min_child_samples=120,
                subsample=0.85,
                colsample_bytree=0.75,
                reg_lambda=1.2,
                objective="multiclass",
                num_class=len(classes),
                random_state=BASE_SEED + stack_fold + 21,
                class_weight=class_weight_idx,
                verbosity=-1,
            )
            lgb_m.fit(X_st_tr, y_st_tr)
            lgb_meta_oof[st_va_idx] = lgb_m.predict_proba(X_st_va)
            lgb_meta_test += lgb_m.predict_proba(meta_test)
        lgb_meta_test /= STACKER_SPLITS
        best_w = 0.0
        best_blend_sc = -1.0
        for w in [0.0, 0.15, 0.3, 0.45, 0.5, 0.55, 0.6, 0.7, 0.85, 1.0]:
            boof = (1.0 - w) * best_stack_proba_oof + w * lgb_meta_oof
            sc = balanced_accuracy_score(
                y, [idx_to_class[i] for i in np.argmax(boof, axis=1)]
            )
            if sc > best_blend_sc:
                best_blend_sc = sc
                best_w = w
        print(
            f"Best LR/LGBM meta blend: w_lgbm={best_w:.2f} OOF balanced_accuracy={best_blend_sc:.6f}"
        )
        best_stack_proba_oof = (1.0 - best_w) * best_stack_proba_oof + best_w * lgb_meta_oof
        best_stack_proba_test = (1.0 - best_w) * best_stack_proba_test + best_w * lgb_meta_test
        best_stack_score = best_blend_sc


    print("Searching temperature scaling on stacker OOF...")

    best_T = 1.0
    best_temp_score = -1.0
    for T in [0.68, 0.72, 0.75, 0.78, 0.82, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1, 1.15, 1.2, 1.3]:
        tp = temper_probs(best_stack_proba_oof, T)
        pred_lbl = [idx_to_class[i] for i in np.argmax(tp, axis=1)]
        sc = balanced_accuracy_score(y, pred_lbl)
        if sc > best_temp_score:
            best_temp_score = sc
            best_T = T

    print(f"Best temperature T={best_T} OOF balanced_accuracy={best_temp_score:.6f}")

    oof_after_temp = temper_probs(best_stack_proba_oof, best_T)
    test_after_temp = temper_probs(best_stack_proba_test, best_T)

    print("Starting class-bias tuning...")

    best_bias_score = -1.0
    best_bias = np.zeros(len(classes), dtype=np.float64)

    bias_checks = 0
    for b_low in np.arange(-0.06, 0.061, 0.01):
        for b_med in np.arange(-0.06, 0.061, 0.01):
            for b_high in np.arange(-0.06, 0.061, 0.01):
                bias = np.array([b_low, b_med, b_high], dtype=np.float64)
                biased_oof = oof_after_temp + bias
                pred_idx = np.argmax(biased_oof, axis=1)
                pred_labels = [idx_to_class[i] for i in pred_idx]
                score = balanced_accuracy_score(y, pred_labels)
                bias_checks += 1
                if bias_checks % 200 == 0:
                    print(f"Bias search progress: checked {bias_checks} combos")
                if score > best_bias_score:
                    best_bias_score = score
                    best_bias = bias.copy()

    oof_best = oof_after_temp + best_bias
    oof_best_pred = np.argmax(oof_best, axis=1)
    oof_best_pred = [idx_to_class[i] for i in oof_best_pred]

    test_proba = test_after_temp + best_bias

    cv_score = balanced_accuracy_score(y, oof_best_pred)
    print("\nCV balanced accuracy (OOF):", round(cv_score, 6))
    print("Fold equal-blend scores:", [round(s, 6) for s in fold_scores])
    print("Mean equal-blend score:", round(float(np.mean(fold_scores)), 6))
    print(f"Best OOF score from stacking search: {best_stack_score:.6f}")
    print(f"Best temperature used: {best_T}")
    bias_text = ", ".join(
        [f"{cls}={offset:+.3f}" for cls, offset in zip(classes, best_bias)]
    )
    print(f"Best class bias offsets: {bias_text}")
    print(f"Best OOF score after bias tuning: {best_bias_score:.6f}")
    print(f"Total bias combos checked: {bias_checks}")
    print(f"Phase run time: {(time.perf_counter() - run_start) / 60.0:.2f} minutes")

    return {
        "test_proba": test_proba,
        "test_after_temp": test_after_temp,
        "cv_score": cv_score,
        "oof_bias_score": best_bias_score,
    }


total_t0 = time.perf_counter()
print("Starting ensemble CV run...")
print(
    f"ORIGINAL_ROW_WEIGHT={ORIGINAL_ROW_WEIGHT} | "
    f"USE_PSEUDO_LABEL={USE_PSEUDO_LABEL} | "
    f"PSEUDO_CONF_MIN={PSEUDO_CONF_MIN}"
)

out1 = run_training_phase(TRAIN_DF_BASE.copy(), "phase1")
test_proba = out1["test_proba"]

if USE_PSEUDO_LABEL:
    train_aug = build_pseudo_augmented_train(TRAIN_DF_BASE, test_df, out1["test_after_temp"])
    out2 = run_training_phase(train_aug, "phase2")
    test_proba = out2["test_proba"]

print(f"\nTotal wall time: {(time.perf_counter() - total_t0) / 60.0:.2f} minutes")


Starting ensemble CV run...
ORIGINAL_ROW_WEIGHT=0.38 | USE_PSEUDO_LABEL=True | PSEUDO_CONF_MIN=0.989

==================== phase1 ====================
Config: OVERNIGHT_MODE=False | seeds=3 | n_splits=5 | stacker_splits=5 | HGB=True ET=False RF=True CAT2=True MLP=True LGBM_META=True

######## Seed 1/3 (seed=42) ########

=== Seed 42 | Fold 1/5 started ===
Seed 42 Fold 1: training CatBoost...
0:	learn: 0.9435616	test: 0.7982221	best: 0.7982221 (0)	total: 1.1s	remaining: 12m 52s
100:	learn: 0.9767154	test: 0.9192667	best: 0.9192667 (100)	total: 1m 22s	remaining: 8m 8s
200:	learn: 0.9787963	test: 0.9239469	best: 0.9239469 (200)	total: 2m 37s	remaining: 6m 30s
300:	learn: 0.9823241	test: 0.9313572	best: 0.9313572 (300)	total: 3m 58s	remaining: 5m 16s
400:	learn: 0.9853534	test: 0.9383963	best: 0.9383963 (400)	total: 5m 22s	remaining: 4m
500:	learn: 0.9872803	test: 0.9441315	best: 0.9441315 (500)	total: 6m 51s	remaining: 2m 43s
600:	learn: 0.9887007	test: 0.9482201	best: 0.9482201 (600)	tot

c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python313\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\ol1v3_7dwns5u\AppData\Local\Programs\Python\Python3

Training until validation scores don't improve for 80 rounds
Did not meet early stopping. Best iteration is:
[800]	valid_0's multi_logloss: 0.066116
Seed 42 Fold 1: training XGBoost...
Seed 42 Fold 1: training HistGradientBoosting...
Seed 42 Fold 1: training MLP...
Seed 42 Fold 1: training RandomForest...
Seed 42 Fold 1: model training complete
Seed 42 Fold 1: cat=0.952713 lgb=0.968306 xgb=0.966609 cat2=0.942614 hgb=0.955103 mlp=0.956672 rf=0.964716
Seed 42 Fold 1: equal-blend balanced_accuracy = 0.969687
Seed 42 Fold 1: completed in 22.73 minutes

=== Seed 42 | Fold 2/5 started ===
Seed 42 Fold 2: training CatBoost...
0:	learn: 0.9381408	test: 0.7345581	best: 0.7345581 (0)	total: 654ms	remaining: 7m 37s
100:	learn: 0.9771195	test: 0.9210511	best: 0.9210977 (99)	total: 1m 18s	remaining: 7m 47s
200:	learn: 0.9794369	test: 0.9258661	best: 0.9258661 (199)	total: 2m 36s	remaining: 6m 27s
300:	learn: 0.9827714	test: 0.9315417	best: 0.9315417 (300)	total: 3m 59s	remaining: 5m 17s
400:	learn:

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X d

Best LR/LGBM meta blend: w_lgbm=0.00 OOF balanced_accuracy=0.971500
Searching temperature scaling on stacker OOF...
Best temperature T=0.68 OOF balanced_accuracy=0.971500
Starting class-bias tuning...
Bias search progress: checked 200 combos
Bias search progress: checked 400 combos
Bias search progress: checked 600 combos
Bias search progress: checked 800 combos
Bias search progress: checked 1000 combos
Bias search progress: checked 1200 combos
Bias search progress: checked 1400 combos
Bias search progress: checked 1600 combos
Bias search progress: checked 1800 combos
Bias search progress: checked 2000 combos

CV balanced accuracy (OOF): 0.971507
Fold equal-blend scores: [0.969687, 0.969837, 0.968793, 0.969659, 0.966618, 0.969672, 0.969157, 0.968326, 0.966555, 0.971668, 0.968204, 0.966657, 0.970577, 0.969194, 0.96836]
Mean equal-blend score: 0.968864
Best OOF score from stacking search: 0.971500
Best temperature used: 0.68
Best class bias offsets: High=-0.060, Low=+0.010, Medium=-0.060

C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\ol1v3_7dwns5u\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2749: UserWarning: X d

Best LR/LGBM meta blend: w_lgbm=0.15 OOF balanced_accuracy=0.975251
Searching temperature scaling on stacker OOF...
Best temperature T=0.68 OOF balanced_accuracy=0.975251
Starting class-bias tuning...
Bias search progress: checked 200 combos
Bias search progress: checked 400 combos
Bias search progress: checked 600 combos
Bias search progress: checked 800 combos
Bias search progress: checked 1000 combos
Bias search progress: checked 1200 combos
Bias search progress: checked 1400 combos
Bias search progress: checked 1600 combos
Bias search progress: checked 1800 combos
Bias search progress: checked 2000 combos

CV balanced accuracy (OOF): 0.975364
Fold equal-blend scores: [0.974006, 0.974124, 0.973183, 0.972534, 0.972316, 0.973243, 0.97368, 0.973363, 0.973654, 0.973897, 0.973565, 0.974741, 0.972609, 0.973458, 0.971908]
Mean equal-blend score: 0.973352
Best OOF score from stacking search: 0.975251
Best temperature used: 0.68
Best class bias offsets: High=-0.060, Low=+0.060, Medium=+0.030

In [6]:
test_proba_blend = test_proba.copy()
if ANCHOR_BLEND_WEIGHT > 0 and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)
    if anchor_labels.isna().any():
        raise ValueError("Anchor blend file is missing ids from test set")
    anchor_idx = anchor_labels.map(class_to_idx).astype(int).values
    anchor_oh = np.zeros_like(test_proba_blend, dtype=np.float64)
    anchor_oh[np.arange(len(anchor_oh)), anchor_idx] = 1.0
    w = float(ANCHOR_BLEND_WEIGHT)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * anchor_oh
    print(f"Anchor probability blend applied (weight={w})")
else:
    print("Anchor probability blend skipped (weight=0 or file missing)")

for path, wt in SUBMISSION_EXTRA_BLENDS:
    if wt <= 0:
        continue
    if not path.exists():
        print(f"Extra submission blend skipped (missing): {path}")
        continue
    sub_df = pd.read_csv(path)[[ID_COL, TARGET]].copy()
    amap = sub_df.set_index(ID_COL)[TARGET]
    alabels = test_df[ID_COL].map(amap)
    if alabels.isna().any():
        raise ValueError(f"Extra blend file missing test ids: {path}")
    aidx = alabels.map(class_to_idx).astype(int).values
    aoh = np.zeros_like(test_proba_blend, dtype=np.float64)
    aoh[np.arange(len(aoh)), aidx] = 1.0
    w = float(wt)
    test_proba_blend = (1.0 - w) * test_proba_blend + w * aoh
    print(f"Extra submission blend: {path.name} weight={w}")

test_conf = np.max(test_proba_blend, axis=1)
test_pred_idx = np.argmax(test_proba_blend, axis=1)
test_pred = pd.Series([idx_to_class[i] for i in test_pred_idx], index=test_df.index)

if USE_ANCHOR_FALLBACK and ANCHOR_SUB_PATH.exists():
    anchor_df = pd.read_csv(ANCHOR_SUB_PATH)
    anchor_df = anchor_df[[ID_COL, TARGET]].copy()
    anchor_map = anchor_df.set_index(ID_COL)[TARGET]
    anchor_labels = test_df[ID_COL].map(anchor_map)

    if anchor_labels.isna().any():
        raise ValueError("Anchor submission is missing ids from test set")

    use_anchor_mask = test_conf < FALLBACK_CONF_THRESHOLD
    test_pred.loc[use_anchor_mask] = anchor_labels.loc[use_anchor_mask]
    print(
        f"Anchor fallback applied to {int(use_anchor_mask.sum())} rows "
        f"(threshold={FALLBACK_CONF_THRESHOLD})"
    )
else:
    print("Anchor fallback disabled or anchor file missing")

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_pred.values,
})
submission.to_csv(SUB_PATH, index=False)

print("Saved:", SUB_PATH.resolve())
submission.head()


Anchor probability blend applied (weight=0.05)
Anchor fallback disabled or anchor file missing
Saved: C:\Users\ol1v3_7dwns5u\OneDrive\Desktop\ACTIVE\s6e4-predicting-irrigation-need\submission.csv


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
